# 🏋️ Notebook 04 — Fine-Tuning the YOLO Detector

**CO543/CO5430 — Traffic Sign Detection**

**Goal**: Fine-tune YOLOv8n on GTSDB using transfer learning from COCO-pretrained weights.

**M3 Deliverable**: Working end-to-end trained detector with preliminary mAP numbers + at least one ablation started.

> 💡 **Tip**: Run this notebook on Google Colab or Kaggle with a free GPU for faster training.  
> Nano (n) or Small (s) model variants are recommended to stay within free-tier limits.

In [ ]:
import sys, os
from pathlib import Path

import torch
import yaml
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# ── Check GPU ──────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — training will be slow. Consider using Colab/Kaggle.')

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'gtsdb_yolov8n.yaml'
print(f'Config  : {CONFIG_PATH}')
print(f'Exists  : {CONFIG_PATH.exists()}')

## 1. Review Training Configuration

In [ ]:
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('── Training Configuration ──')
for k, v in cfg.items():
    print(f'  {k:20s}: {v}')

## 2. Run Training

In [ ]:
# ── Training parameters (override config values here if needed) ────────
MODEL_WEIGHTS = 'yolov8n.pt'   # COCO pretrained weights
EPOCHS        = 50
IMG_SIZE      = 640
BATCH_SIZE    = 16             # Reduce to 8 if GPU runs out of memory
RUN_NAME      = 'gtsdb_yolov8n_v1'

print(f'Starting training: {MODEL_WEIGHTS} for {EPOCHS} epochs on device={DEVICE}')
print('This will take ~30–60 min on a GPU, several hours on CPU.')
print()

model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data    = str(CONFIG_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    name    = RUN_NAME,
    project = str(PROJECT_ROOT / 'runs' / 'train'),
    exist_ok= True,
    # Augmentation overrides (fliplr=0 is critical for direction-sensitive signs)
    fliplr  = 0.0,
    degrees = 15.0,
    hsv_v   = 0.4,
    hsv_s   = 0.7,
)

print('\n✅ Training complete!')
BEST_WEIGHTS = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'Best weights saved at: {BEST_WEIGHTS}')

## 3. Plot Training Curves

In [ ]:
run_dir = PROJECT_ROOT / 'runs' / 'train' / RUN_NAME
csv_path = run_dir / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()  # remove leading spaces from column names
    print('Columns:', df.columns.tolist())

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f'Training Curves — {RUN_NAME}', fontsize=14)

    metrics_to_plot = [
        ('train/box_loss',  'Train Box Loss',  'red'),
        ('train/cls_loss',  'Train Cls Loss',  'orange'),
        ('val/box_loss',    'Val Box Loss',    'blue'),
        ('val/cls_loss',    'Val Cls Loss',    'purple'),
        ('metrics/mAP50',   'mAP@0.5',         'green'),
        ('metrics/mAP50-95','mAP@0.5:0.95',    'teal'),
    ]

    for ax, (col, label, colour) in zip(axes.flatten(), metrics_to_plot):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=colour, lw=2)
            ax.set_title(label, fontsize=11)
            ax.set_xlabel('Epoch')
            ax.grid(alpha=0.3)
        else:
            ax.set_title(f'{label} (not found)', fontsize=11)

    plt.tight_layout()
    out = PROJECT_ROOT / 'results' / 'figures' / f'training_curves_{RUN_NAME}.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print(f'Saved → {out}')
else:
    print(f'results.csv not found at {csv_path} — run training first.')

## 4. Quick Validation Check After Training

In [ ]:
BEST_WEIGHTS = PROJECT_ROOT / 'runs' / 'train' / RUN_NAME / 'weights' / 'best.pt'

if BEST_WEIGHTS.exists():
    best_model = YOLO(str(BEST_WEIGHTS))
    val_results = best_model.val(
        data  = str(CONFIG_PATH),
        split = 'val',
        device= DEVICE,
    )
    print('\n── Validation Results (Best Checkpoint) ──')
    print(f'  mAP@0.5      : {val_results.box.map50:.4f}')
    print(f'  mAP@0.5:0.95 : {val_results.box.map:.4f}')
    print(f'  Precision    : {val_results.box.mp:.4f}')
    print(f'  Recall       : {val_results.box.mr:.4f}')
else:
    print(f'Best weights not found at {BEST_WEIGHTS}. Complete training first.')

## 5. Ablation — With vs Without Augmentation

*Run this after the main training above. Trains a second model variant.*

In [ ]:
# ── Ablation: no augmentation ──────────────────────────────────────────
RUN_NAME_NO_AUG = 'gtsdb_yolov8n_noaug_v1'

model_no_aug = YOLO(MODEL_WEIGHTS)
results_no_aug = model_no_aug.train(
    data    = str(CONFIG_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    name    = RUN_NAME_NO_AUG,
    project = str(PROJECT_ROOT / 'runs' / 'train'),
    exist_ok= True,
    # Disable all augmentation
    fliplr  = 0.0, flipud = 0.0, degrees = 0.0,
    hsv_h   = 0.0, hsv_s  = 0.0, hsv_v   = 0.0,
    mosaic  = 0.0, mixup  = 0.0, scale   = 0.0,
    translate = 0.0,
)

print('Ablation (no augmentation) training complete.')

## 6. Ablation — Model Size Comparison (nano vs small)

*Optional: trains YOLOv8s for the model-size ablation.*

In [ ]:
RUN_NAME_SMALL = 'gtsdb_yolov8s_v1'

model_small = YOLO('yolov8s.pt')   # Small variant (~11 MB vs ~6 MB nano)
results_small = model_small.train(
    data    = str(CONFIG_PATH),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = 8,          # smaller batch for larger model
    device  = DEVICE,
    name    = RUN_NAME_SMALL,
    project = str(PROJECT_ROOT / 'runs' / 'train'),
    exist_ok= True,
    fliplr  = 0.0,
    degrees = 15.0,
)

print('YOLOv8s training complete.')

## 7. Copy Best Checkpoint to results/

For easy access during evaluation and the demo.

In [ ]:
import shutil

checkpoints_dir = PROJECT_ROOT / 'results' / 'checkpoints'
checkpoints_dir.mkdir(parents=True, exist_ok=True)

for run_name in [RUN_NAME, RUN_NAME_NO_AUG, RUN_NAME_SMALL]:
    best = PROJECT_ROOT / 'runs' / 'train' / run_name / 'weights' / 'best.pt'
    if best.exists():
        dest = checkpoints_dir / f'{run_name}_best.pt'
        shutil.copy2(best, dest)
        print(f'Copied: {dest}')
    else:
        print(f'Not found (training may not be complete): {best}')

print('\n⚠️  Remember: do NOT commit .pt files to GitHub. Provide a download link instead.')